# Colab TTS Quality Gate — clean isolated environments

This notebook deliberately does **not** install Qwen and Chatterbox into the Colab kernel. The previous report showed `torch 2.11.0+cu128` mixed with package files and failed inside `torch._C._dynamo` (`skip_code`). That means the runtime environment was contaminated by package changes.

Each model is installed and executed in its own Python virtual environment from a subprocess, so the notebook kernel's PyTorch is never modified. This also avoids the Qwen/Chatterbox dependency conflict.


In [ ]:
# GPU preflight — IMPORT ONLY the Colab kernel's torch. No pip yet.
import sys,torch
print('Python:',sys.version)
print('Kernel torch:',torch.__version__)
print('Kernel CUDA:',torch.version.cuda)
if not torch.cuda.is_available(): raise RuntimeError('GPU unavailable. Select Runtime > Change runtime type > GPU.')
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))

## Session A — Qwen3-TTS

The installer below creates a fresh venv. It uses a matched PyTorch 2.6 CUDA 12.6 wheel inside that venv, rather than touching Colab's PyTorch. Qwen's own package documentation recommends a fresh isolated environment. citeturn1view0

In [ ]:
# Build a completely isolated Qwen environment.
# Safe to run even if the Colab kernel has broken/mixed torch packages.
import os,sys,subprocess
VENV='/content/tts_qwen_env'
if not os.path.exists(VENV+'/bin/python'):
    subprocess.check_call([sys.executable,'-m','venv',VENV])
PY=VENV+'/bin/python'; PIP=[PY,'-m','pip']
subprocess.check_call(PIP+['install','-q','--upgrade','pip'])
subprocess.check_call(PIP+['install','-q','--only-binary=:all:','numpy==1.26.4'])
subprocess.check_call(PIP+['install','-q','torch==2.6.0','torchvision==0.21.0','torchaudio==2.6.0','--index-url','https://download.pytorch.org/whl/cu126'])
subprocess.check_call(PIP+['install','-q','--no-cache-dir','qwen-tts==0.1.1','soundfile','scipy'])
print('Qwen isolated environment ready:',PY)

In [ ]:
# Execute Qwen in the isolated interpreter — no imports of qwen/torch from the notebook kernel.
from pathlib import Path
import subprocess,textwrap,json
out=Path('/content/openmontage-colab/projects/colab-tts-quality'); out.mkdir(parents=True,exist_ok=True)
script=out/'run_qwen_isolated.py'
script.write_text(r'''import json,time,traceback
from pathlib import Path
import torch,numpy as np,soundfile as sf
from qwen_tts import Qwen3TTSModel
OUT=Path('/content/openmontage-colab/projects/colab-tts-quality'); WAV=OUT/'qwen3_tts_output.wav'; REPORT=OUT/'qwen_report.json'
text='A hundred years ago, humanity looked toward the stars and wondered whether we were alone. Tonight, something answered. The signal came from a world no telescope had ever seen before. And buried inside that transmission was a message meant for us.'
instruction='Calm cinematic documentary narration. Natural pacing, clear pronunciation, subtle mystery and anticipation, and natural emotional expression.'
r={'model':'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice','status':'NOT_RUN','torch':torch.__version__,'cuda':torch.version.cuda}
try:
 t=time.time(); model=Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice',device_map='cuda:0',dtype=torch.bfloat16); r['load_seconds']=round(time.time()-t,2)
 t=time.time(); wavs,sr=model.generate_custom_voice(text=text,language='English',speaker='Ryan',instruct=instruction); sf.write(str(WAV),np.asarray(wavs[0]),int(sr)); r.update({'generation_seconds':round(time.time()-t,2),'sample_rate':int(sr),'file_size':WAV.stat().st_size,'status':'REAL_PASS' if WAV.stat().st_size>1000 else 'REAL_FAIL'})
except Exception as e: r.update({'status':'REAL_FAIL','error':str(e),'traceback':traceback.format_exc()})
REPORT.write_text(json.dumps(r,indent=2,default=str)); print(json.dumps(r,indent=2,default=str))
''',encoding='utf-8')
res=subprocess.run(['/content/tts_qwen_env/bin/python',str(script)],capture_output=True,text=True)
print(res.stdout)
if res.returncode!=0: print(res.stderr)
if res.returncode!=0: raise RuntimeError('Qwen subprocess failed; see traceback above.')

## Session B — Chatterbox-Turbo

Run this section in the same Colab runtime. It uses a second venv, so its `torch==2.6.0` and `transformers` cannot contaminate the Qwen environment or the notebook kernel. Chatterbox's official usage also requires a reference audio clip for Turbo generation. citeturn0search1

In [ ]:
# Create an isolated Chatterbox environment.
import os,sys,subprocess
VENV='/content/tts_chatterbox_env'
if not os.path.exists(VENV+'/bin/python'):
    subprocess.check_call([sys.executable,'-m','venv',VENV])
PY=VENV+'/bin/python'; PIP=[PY,'-m','pip']
subprocess.check_call(PIP+['install','-q','--upgrade','pip'])
subprocess.check_call(PIP+['install','-q','--only-binary=:all:','numpy==1.26.4'])
subprocess.check_call(PIP+['install','-q','torch==2.6.0','torchvision==0.21.0','torchaudio==2.6.0','--index-url','https://download.pytorch.org/whl/cu126'])
subprocess.check_call(PIP+['install','-q','--no-cache-dir','chatterbox-tts==0.1.7','soundfile','scipy'])
print('Chatterbox isolated environment ready:',PY)

In [ ]:
# Upload a short clean reference voice WAV. Chatterbox Turbo officially expects a reference clip.
from google.colab import files
uploaded=files.upload()
if not uploaded: raise RuntimeError('Upload a reference WAV before running Chatterbox.')
ref_name=next(iter(uploaded)); print('Reference:',ref_name)

In [ ]:
from pathlib import Path
import subprocess
out=Path('/content/openmontage-colab/projects/colab-tts-quality'); out.mkdir(parents=True,exist_ok=True)
script=out/'run_chatterbox_isolated.py'
script.write_text(r'''import json,time,traceback,sys
from pathlib import Path
import torch
from chatterbox.tts_turbo import ChatterboxTurboTTS
import torchaudio
OUT=Path('/content/openmontage-colab/projects/colab-tts-quality'); WAV=OUT/'chatterbox_output.wav'; REPORT=OUT/'chatterbox_report.json'
REF=sys.argv[1]
text='A hundred years ago, humanity looked toward the stars and wondered whether we were alone. Tonight, something answered. The signal came from a world no telescope had ever seen before. And buried inside that transmission was a message meant for us.'
r={'model':'ResembleAI/chatterbox-turbo','status':'NOT_RUN','torch':torch.__version__,'cuda':torch.version.cuda}
try:
 t=time.time(); model=ChatterboxTurboTTS.from_pretrained(device='cuda'); r['load_seconds']=round(time.time()-t,2)
 t=time.time(); wav=model.generate(text,audio_prompt_path=REF); torchaudio.save(str(WAV),wav.cpu(),model.sr); r.update({'generation_seconds':round(time.time()-t,2),'sample_rate':int(model.sr),'file_size':WAV.stat().st_size,'status':'REAL_PASS' if WAV.stat().st_size>1000 else 'REAL_FAIL'})
except Exception as e: r.update({'status':'REAL_FAIL','error':str(e),'traceback':traceback.format_exc()})
REPORT.write_text(json.dumps(r,indent=2,default=str)); print(json.dumps(r,indent=2,default=str))
''',encoding='utf-8')
res=subprocess.run(['/content/tts_chatterbox_env/bin/python',str(script),'/content/'+ref_name],capture_output=True,text=True)
print(res.stdout)
if res.returncode!=0: print(res.stderr)
if res.returncode!=0: raise RuntimeError('Chatterbox subprocess failed; see traceback above.')

In [ ]:
# Final report + audio playback.
from pathlib import Path
from IPython.display import Audio,display
import json
out=Path('/content/openmontage-colab/projects/colab-tts-quality')
for p in [out/'qwen_report.json',out/'chatterbox_report.json']:
    if p.exists(): print(p.name, p.read_text())
for p in [out/'qwen3_tts_output.wav',out/'chatterbox_output.wav']:
    if p.exists() and p.stat().st_size>1000: print('Audio:',p); display(Audio(str(p)))